# 09 - Imputation Analysis
Evaluate imputation quality: Age distribution preservation, KS test, Wasserstein distance, and model metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

df = pd.read_parquet('data/processed/workforce_clean_base.parquet')
df_raw = pd.read_csv('data/raw/employee_data.csv')
print(f'Raw: {len(df_raw)} rows, Processed: {len(df)} rows')

In [ ]:
# Age imputation analysis
print('=== Age Imputation ===')
has_age_raw = df_raw['Age'].notna().sum()
total = len(df_raw)
imputed_count = total - has_age_raw
print(f'Original Age available: {has_age_raw} ({has_age_raw/total:.1%})')
print(f'Age imputed: {imputed_count} ({imputed_count/total:.1%})')
print(f'After pipeline - Age missing: {df["Age"].isna().sum()} (should be 0)')

In [ ]:
# Distribution comparison
original_age = df_raw['Age'].dropna()
final_age = df['Age']

# KS test
ks_stat, ks_p = stats.ks_2samp(original_age, final_age)
print(f'KS statistic: {ks_stat:.4f}')
print(f'KS p-value: {ks_p:.4f}')
print(f'Wasserstein distance: {stats.wasserstein_distance(original_age, final_age):.4f}')
print(f'\nOriginal mean: {original_age.mean():.1f}, Final mean: {final_age.mean():.1f}')
print(f'Original std: {original_age.std():.1f}, Final std: {final_age.std():.1f}')

fig, ax = plt.subplots(figsize=(10, 4))
original_age.plot(kind='hist', bins=20, alpha=0.6, label='Original (known)', ax=ax)
final_age.plot(kind='hist', bins=20, alpha=0.6, label='After imputation', ax=ax)
ax.set_xlabel('Age')
ax.set_title(f'Age Distribution: KS p={ks_p:.3f}')
ax.legend()
plt.tight_layout()

In [ ]:
# Check other critical fields
print('=== Field Completeness ===')
for col in ['Age', 'GenderCode', 'DepartmentType', 'Performance Score', 'TenureYears', 'EmployeeStatus']:
    if col in df.columns:
        missing = df[col].isna().sum()
        print(f'  {col}: {missing} missing ({missing/len(df):.1%})')

In [ ]:
print('=== Imputation Summary ===')
print('Ensemble imputation: 20% median + 40% RF + 40% GBM')
print('Distribution-matching: samples from known values to preserve variance')
print(f'Target metric: KS p-value > 0.05 (target: 0.91)')